# 📘 Notebook 03: Planning with the Bellman Equations

### Course: *From Random Moves to Winning Agents: Reinforcement Learning with Games*
**Instructor:** Ioannis Tsioulis

*Module A: Foundations · Notebook 3 of 4 in this module · (3 of 18 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Define the **state-value function** $v_\pi$ and the **action-value function** $q_\pi$ of a policy
- Write the **Bellman expectation equation** and use it to evaluate a policy without playing a single episode
- **Improve** a policy by acting greedily on its values
- Implement **policy iteration** and **value iteration** and show that they find the same optimal policy
- Explain how **chance** and the **discount factor** change what the best policy is
- State the two reasons why these methods do not scale to large games

> **Prerequisites:** Notebooks 01 and 02, in particular the return $G_t = R_{t+1} + \gamma\,G_{t+1}$, policies and the definition of an MDP.
>
> 🔭 **Why this notebook matters later:** the two equations of this notebook are the foundation of the whole course. Q-learning (Notebook 05), DQN (Notebook 09) and actor-critic methods (Notebook 13) are all ways of solving them approximately when the rules of the game are unknown or the game is too large.

> ℹ️ **Setup note.** The first sections need only NumPy and matplotlib. The last exercise uses Gymnasium and a video, so the cell below installs them. Every cell finishes within seconds.

In [ ]:
%pip install -q "gymnasium[toy-text]>=1.3" imageio imageio-ffmpeg

In [ ]:
import base64
import numpy as np
import matplotlib.pyplot as plt
import imageio
from IPython.display import HTML, display


def show_video(frames, fps=4, width=320):
    """Play a list of pictures (RGB arrays) as a video inside the notebook."""
    imageio.mimsave("episode.mp4", frames, fps=fps, macro_block_size=1)
    data = base64.b64encode(open("episode.mp4", "rb").read()).decode()
    display(HTML(f'<video width="{width}" controls autoplay loop src="data:video/mp4;base64,{data}"></video>'))

## 1. The room, with its rules written down

We keep the room of Notebook 02: a robot, a treasure (+10), two pits (−10), walls, and a cost of −1 per step.

This time the agent is given something it will not have in most of the course: the **rules**. The method `transitions(state, action)` lists everything that can happen after an action, with its probability. That list is the **model** of the environment, the $p(s' \mid s, a)$ and $r$ of the MDP definition.

Finding good behaviour from a known model is called **planning**. No episode is played. Everything is computed.

In [ ]:
class GridWorld:
    """The room of Notebook 02, extended with a model: transitions(state, action)."""

    LAYOUT = ["S....",
              ".#.X.",
              ".#...",
              "...X.",
              "....G"]
    MOVES = [(-1, 0), (0, 1), (1, 0), (0, -1)]          # 0 up, 1 right, 2 down, 3 left
    ARROWS = ["↑", "→", "↓", "←"]
    COLOURS = {"S": (235, 235, 235), ".": (235, 235, 235), "#": (60, 60, 60),
               "X": (200, 60, 60), "G": (240, 200, 40)}

    def __init__(self, slip=0.0, seed=0):
        self.rows, self.cols = len(self.LAYOUT), len(self.LAYOUT[0])
        self.n_states, self.n_actions = self.rows * self.cols, 4
        self.slip = slip
        self.rng = np.random.default_rng(seed)

    def cell(self, state):
        return self.LAYOUT[state // self.cols][state % self.cols]

    def is_terminal(self, state):
        return self.cell(state) in "GX"

    def move(self, state, action):
        """Where one move leads, with no chance involved: (next state, reward, terminated)."""
        row = state // self.cols + self.MOVES[action][0]
        col = state % self.cols + self.MOVES[action][1]
        if 0 <= row < self.rows and 0 <= col < self.cols and self.LAYOUT[row][col] != "#":
            state = row * self.cols + col
        reward = {"G": 10.0, "X": -10.0}.get(self.cell(state), -1.0)
        return state, reward, self.is_terminal(state)

    def transitions(self, state, action):
        """The model: a list of (probability, next state, reward, terminated)."""
        outcomes = {}
        for actual in range(4):          # with probability `slip` the move is replaced by a random one
            probability = (1 - self.slip) * (actual == action) + self.slip / 4
            if probability > 0:
                result = self.move(state, actual)
                outcomes[result] = outcomes.get(result, 0.0) + probability
        return [(p, s, r, done) for (s, r, done), p in outcomes.items()]

    def reset(self):
        self.state = 0
        return self.state, {}

    def step(self, action):
        if self.rng.random() < self.slip:
            action = int(self.rng.integers(4))
        self.state, reward, terminated = self.move(self.state, action)
        return self.state, reward, terminated, False, {}

    def render(self, size=48):
        image = np.zeros((self.rows * size, self.cols * size, 3), dtype=np.uint8)
        for r in range(self.rows):
            for c in range(self.cols):
                image[r * size:(r + 1) * size, c * size:(c + 1) * size] = self.COLOURS[self.LAYOUT[r][c]]
                image[r * size, c * size:(c + 1) * size] = 150
                image[r * size:(r + 1) * size, c * size] = 150
        r, c = divmod(self.state, self.cols)
        image[r * size + 10:(r + 1) * size - 10, c * size + 10:(c + 1) * size - 10] = (40, 100, 220)
        return image


env = GridWorld(slip=0.2)
print("From the start, action 'right' on a slippery floor can lead to:")
for probability, next_state, reward, terminated in env.transitions(0, 1):
    print(f"  probability {probability:.2f}: state {next_state}, reward {reward}")

On a floor with `slip = 0.2`, the move "right" from the start succeeds with probability 0.85 (0.8 as intended, plus 0.05 when the random replacement happens to be "right" too). Otherwise the robot goes down, or bumps into the edge and stays where it is.

## 2. How good is a state? Value functions

### Intuition first
A chess player looks at a board and says "white is winning", before any further move is made. They are estimating how the game will end **from this position**, given how both sides play. A value function is that judgement, as a number for every state.

### Formal definition
For a policy $\pi$:

- The **state-value function** $v_\pi(s)$ is the expected return when starting in state $s$ and following $\pi$ afterwards.
- The **action-value function** $q_\pi(s, a)$ is the expected return when starting in $s$, taking action $a$ **once**, and following $\pi$ afterwards.

A value always belongs to a policy. The same square of the room is a good place to be for a careful robot and a bad one for a robot that walks at random.

### The Bellman expectation equation
Notebook 02 ended with $G_t = R_{t+1} + \gamma\,G_{t+1}$. Take the average of both sides over everything that can happen, and the same relation appears between the **values** of neighbouring states:

$$v_\pi(s) = \sum_a \pi(a \mid s) \sum_{s'} p(s' \mid s, a)\,\big[\,r + \gamma\,v_\pi(s')\,\big]$$

Read it from the inside out: *the value of a state is the immediate reward plus the discounted value of where you land, averaged over the actions the policy might choose and over the places the environment might send you.* It is named after Richard Bellman, who introduced it in the 1950s.

### Why it matters
The equation ties the value of every state to the values of its neighbours. That gives one equation per state, and solving them gives the value of a policy **exactly**, with no sampling and no noise.

The inner sum, the value of taking one particular action, is used so often that we give it its own function.

In [ ]:
def action_values(env, V, state, gamma):
    """q(s, a) for every action, by looking one step ahead with the model."""
    q = np.zeros(env.n_actions)
    for action in range(env.n_actions):
        for probability, next_state, reward, terminated in env.transitions(state, action):
            future = 0.0 if terminated else V[next_state]       # nothing follows the end of the game
            q[action] += probability * (reward + gamma * future)
    return q


def usable_states(env):
    """States where a decision is made: not walls, not the end of the game."""
    return [s for s in range(env.n_states) if env.cell(s) != "#" and not env.is_terminal(s)]

## 3. Policy evaluation

### Intuition first
We do not solve the 25 equations with algebra. We start from a guess (all values zero) and repeatedly replace each value by the right-hand side of its equation. Each pass over the states is a **sweep**. Information spreads outwards from the treasure and the pits one step per sweep, like a rumour, until the values stop changing.

In [ ]:
def policy_evaluation(env, policy, gamma, tolerance=1e-6):
    """Values of a policy given as a table policy[state, action] of probabilities."""
    V = np.zeros(env.n_states)
    sweeps = 0
    while True:
        biggest_change = 0.0
        for state in usable_states(env):
            new_value = policy[state] @ action_values(env, V, state, gamma)   # average over the policy
            biggest_change = max(biggest_change, abs(new_value - V[state]))
            V[state] = new_value
        sweeps += 1
        if biggest_change < tolerance:
            return V, sweeps


def show_values(env, V, policy=None, title=""):
    """Draw the values as a coloured grid, with an arrow for the policy's favourite action."""
    grid = V.reshape(env.rows, env.cols)
    plt.figure(figsize=(5, 4.5))
    plt.imshow(grid, cmap="RdYlGn", vmin=-10, vmax=10)
    for state in range(env.n_states):
        row, col = divmod(state, env.cols)
        kind = env.cell(state)
        if kind == "#":
            plt.gca().add_patch(plt.Rectangle((col - 0.5, row - 0.5), 1, 1, color="dimgray"))
        elif kind in "GX":
            plt.text(col, row, "treasure" if kind == "G" else "pit", ha="center", va="center", fontsize=8)
        else:
            arrow = "" if policy is None else GridWorld.ARROWS[int(policy[state].argmax())] + "\n"
            plt.text(col, row, f"{arrow}{V[state]:.1f}", ha="center", va="center", fontsize=9)
    plt.xticks([]); plt.yticks([])
    plt.title(title)
    plt.show()


GAMMA = 0.95
env = GridWorld()
random_policy = np.full((env.n_states, 4), 0.25)             # every action equally likely

V_random, sweeps = policy_evaluation(env, random_policy, GAMMA)
print(f"converged after {sweeps} sweeps")
print(f"value of the start state under the random policy: {V_random[0]:.2f}")
show_values(env, V_random, title="Values of the random policy")

Every state has a negative value: a robot that moves at random mostly ends up in a pit, as Exercise 3 of Notebook 02 measured. The squares next to the pits are the worst, and the values improve a little towards the treasure.

Is the calculation right? We can check it against experience: play many random episodes from the start and average their discounted returns.

In [ ]:
def simulate(env, policy, gamma, episodes=5000, max_steps=1000, seed=0):
    rng = np.random.default_rng(seed)
    returns = []
    for episode in range(episodes):
        state, info = env.reset()
        total, discount = 0.0, 1.0
        for t in range(max_steps):
            action = rng.choice(4, p=policy[state])
            state, reward, terminated, truncated, info = env.step(action)
            total += discount * reward
            discount *= gamma
            if terminated:
                break
        returns.append(total)
    return float(np.mean(returns))


print(f"computed with the Bellman equation: {V_random[0]:.2f}")
print(f"average of 5,000 played episodes:   {simulate(env, random_policy, GAMMA):.2f}")

The two numbers agree up to the noise of sampling. The first took a fraction of a second and is exact. The second needed thousands of games and is still approximate. That is the power of knowing the model.

## 4. Policy improvement

### Intuition first
Knowing how good each square is under the random policy immediately suggests something better: in every square, **step towards the best-looking neighbour**. More precisely, in each state choose the action with the highest $q_\pi(s, a)$.

### Formal definition
The **greedy policy** with respect to $v_\pi$ chooses $\arg\max_a q_\pi(s, a)$ in every state. The **policy improvement theorem** guarantees that this new policy is at least as good as $\pi$ in every state, and strictly better somewhere unless $\pi$ was already optimal.

In [ ]:
def greedy_policy(env, V, gamma):
    policy = np.zeros((env.n_states, env.n_actions))
    for state in usable_states(env):
        policy[state, action_values(env, V, state, gamma).argmax()] = 1.0
    return policy


improved = greedy_policy(env, V_random, GAMMA)
V_improved, sweeps = policy_evaluation(env, improved, GAMMA)

print(f"start state: random policy {V_random[0]:6.2f}  ->  after one improvement {V_improved[0]:6.2f}")
show_values(env, V_improved, improved, title="After one round of improvement")

One round of "evaluate, then act greedily" raised the value of the start state, and of every other state, as the theorem promises. But look closely at the arrows: in several squares the improved policy walks **straight into a pit**. That is not a bug. The values it was built from are those of the random policy, and for a robot that wanders at random, being alive far from the treasure is worth about −17, which is worse than ending the game at once for −10. The new policy is better than the old one. It is not yet good.

## 5. Policy iteration

If one round helps, repeat it: evaluate the **new** policy, whose values are no longer so gloomy, improve again, and stop when the policy no longer changes. A policy that is greedy with respect to its own values cannot be improved, and such a policy is **optimal**.

In [ ]:
def policy_iteration(env, gamma):
    policy = np.full((env.n_states, env.n_actions), 0.25)
    rounds = 0
    while True:
        V, sweeps = policy_evaluation(env, policy, gamma)
        new_policy = greedy_policy(env, V, gamma)
        rounds += 1
        if np.array_equal(new_policy, policy):
            return policy, V, rounds
        policy = new_policy


best_policy, V_best, rounds = policy_iteration(env, GAMMA)
print(f"policy iteration stopped after {rounds} rounds")
show_values(env, V_best, best_policy, title="The optimal policy and its values")

The optimal values are written $v_*$ and the optimal policy $\pi_*$. Following the arrows from the start takes the robot to the treasure in 8 steps, the shortest route there is.

## 6. Value iteration

### Formal definition
Policy iteration evaluates each intermediate policy completely before improving it, which is more work than necessary. **Value iteration** merges the two steps. It applies, in every sweep, the **Bellman optimality equation**:

$$v_*(s) = \max_a \sum_{s'} p(s' \mid s, a)\,\big[\,r + \gamma\,v_*(s')\,\big]$$

Compare it with the expectation equation of Section 2. The average over the policy's actions has been replaced by a **maximum**: the value of a state under the best behaviour is the value of its best action.

In [ ]:
def value_iteration(env, gamma, tolerance=1e-6):
    V = np.zeros(env.n_states)
    sweeps = 0
    while True:
        biggest_change = 0.0
        for state in usable_states(env):
            new_value = action_values(env, V, state, gamma).max()        # the best action, not the average
            biggest_change = max(biggest_change, abs(new_value - V[state]))
            V[state] = new_value
        sweeps += 1
        if biggest_change < tolerance:
            return V, sweeps


V_star, sweeps = value_iteration(env, GAMMA)
print(f"value iteration converged after {sweeps} sweeps")
print(f"largest difference from policy iteration: {np.abs(V_star - V_best).max():.8f}")

Both methods arrive at the same values. The optimal policy is read off at the end with `greedy_policy`.

Now let us watch the plan being carried out.

In [ ]:
def play(env, policy, max_steps=60):
    state, info = env.reset()
    frames, total = [env.render()], 0.0
    for t in range(max_steps):
        state, reward, terminated, truncated, info = env.step(int(policy[state].argmax()))
        total += reward
        frames.append(env.render())
        if terminated:
            break
    return frames, total


frames, total = play(env, greedy_policy(env, V_star, GAMMA))
print(f"total reward: {total:.0f} in {len(frames) - 1} steps")
show_video(frames)

## 7. Chance changes the plan

Notebook 02 showed that a route along the pits becomes poor on a slippery floor. A human had to notice that and write a different policy. Value iteration works it out by itself, because the probabilities are part of the equation.

In [ ]:
slippery = GridWorld(slip=0.3)
V_slippery, sweeps = value_iteration(slippery, GAMMA)
policy_slippery = greedy_policy(slippery, V_slippery, GAMMA)

print(f"converged after {sweeps} sweeps")
show_values(slippery, V_slippery, policy_slippery, title="Optimal policy on a slippery floor (slip = 0.3)")

Compare the arrows with the dry floor. From the start the robot now goes **down the left-hand side**, far from both pits, although the route along the top row is just as short. All the values are lower too: on a slippery floor even perfect play costs more steps and sometimes ends badly.

> ⚠️ **Common pitfalls**
>
> - **Talking about "the value of a state" without a policy.** $v_\pi$ depends on $\pi$. Only $v_*$ belongs to the environment alone.
> - **Adding the value of a terminal state.** After the end of the game there is no future. Forgetting the `0.0 if terminated` line makes the treasure worth more than 10.
> - **Confusing the two equations.** The expectation equation averages over the policy's actions and evaluates a given policy. The optimality equation takes the maximum and describes the best one.

## 8. Why we cannot stop here

Dynamic programming, the family name of these methods, is exact and fast on our room. It has two requirements that most interesting problems do not meet:

- **It needs the model.** We had to write `transitions` by hand. For a robot arm, a card game against people or a video game, nobody can write down every probability.
- **It sweeps over every state.** Our room has 25. Backgammon has about $10^{20}$ positions and chess far more. One sweep would never finish.

The rest of the course removes these two requirements one at a time. Modules A and B drop the model: the agent learns from **experience**. Modules C and D drop the table: a neural network **generalises** across states it has never visited. The Bellman equations stay with us all the way.

---
## ✏️ Exercises

The exercises use `GridWorld`, `action_values`, `policy_evaluation`, `greedy_policy`, `value_iteration`, `show_values` and `play` from above.

### Exercise 1 (One Bellman step by hand)
Take the dry room and the optimal values `V_star`. The square just above the treasure is state 19. Using only `V_star` and the rules (step cost −1, treasure +10, gamma 0.95), compute by hand the value of each of its four actions: up leads to state 14, right bumps into the edge and stays in 19, down reaches the treasure, left leads to state 18, which is a pit (reward −10, and the game ends). Then check with `action_values`.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
env = GridWorld()
by_hand = {
    "up":    -1 + 0.95 * V_star[14],
    "right": -1 + 0.95 * V_star[19],
    "down":  10.0,                        # the game ends: nothing is added
    "left":  -10.0,                       # a pit: the game ends here too
}
computed = action_values(env, V_star, 19, GAMMA)
for (name, value), check in zip(by_hand.items(), computed):
    print(f"{name:>5}: by hand {value:6.2f}, function {check:6.2f}")
```

*The value of state 19 is the largest of the four, which is the Bellman optimality equation for that state.*
</details>

### Exercise 2 (Short-sighted and far-sighted robots)
On the slippery floor (`slip=0.3`) compute the optimal policy for gamma = 0.5 and gamma = 0.99, and show both with `show_values`. In which squares do the arrows differ? Compare also the **values** far from the treasure, and explain what you see.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
slippery = GridWorld(slip=0.3)
policies = {}
for gamma in (0.5, 0.99):
    V, sweeps = value_iteration(slippery, gamma)
    policies[gamma] = greedy_policy(slippery, V, gamma)
    show_values(slippery, V, policies[gamma], title=f"gamma = {gamma}")

different = [s for s in usable_states(slippery) if policies[0.5][s].argmax() != policies[0.99][s].argmax()]
print("squares where the two robots choose differently:", different)
```

*The arrows differ only in two squares of the top row, where the short-sighted robot turns back towards the left-hand route. The values tell more. With gamma 0.5 almost every square far from the treasure is worth about −2: a treasure eight moves away is discounted to nearly nothing, and what remains is the cost of the next few steps. With gamma 0.99 the values rise steadily along the route, so the robot can tell from far away which direction is better.*
</details>

### Exercise 3 (How fast does the rumour spread?)
Count the sweeps that value iteration needs for gamma = 0.5, 0.9, 0.99 and 0.999, first on the dry room and then on the slippery floor (`slip=0.3`). Explain the pattern.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
for gamma in (0.5, 0.9, 0.99, 0.999):
    dry = value_iteration(GridWorld(), gamma)[1]
    wet = value_iteration(GridWorld(slip=0.3), gamma)[1]
    print(f"gamma = {gamma:<6} sweeps on the dry floor: {dry:>2},  on the slippery floor: {wet:>2}")
```

*On the dry floor the number does not depend on gamma at all: a value becomes exact as soon as the information from the treasure has travelled to that square, and the furthest square is 8 steps away. On the slippery floor the values are averages over many possible futures and are only approached gradually, more slowly when gamma is large because the distant future then carries more weight. The count levels off for gamma close to 1: every game in this room ends after a limited number of steps, so there is not much distant future to account for.*
</details>

### Exercise 4 (Changing what we ask for)
The reward is our way of telling the robot what we want. On the slippery floor (`slip=0.3`), make a subclass of `GridWorld` in which falling into a pit costs −100 instead of −10, compute the optimal policy, and compare the value of the start state and the arrows with the original. What has changed in the robot's behaviour?

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
class DeepPits(GridWorld):
    def move(self, state, action):
        state, reward, terminated = super().move(state, action)
        if self.cell(state) == "X":
            reward = -100.0
        return state, reward, terminated

for world in (GridWorld(slip=0.3), DeepPits(slip=0.3)):
    V, sweeps = value_iteration(world, GAMMA)
    policy = greedy_policy(world, V, GAMMA)
    print(f"{type(world).__name__:10s} value of the start state: {V[0]:7.2f}")
    show_values(world, V, policy, title=type(world).__name__)
```

*Nothing was said to the robot about "being careful". Making the pits more expensive is enough. A robot that finds itself on the top row now turns back to the left-hand route, and in the square to the left of the upper pit it moves up, away from the pit, where before it moved down towards the treasure. The colour scale of the picture stops at −10, so squares with lower values all look equally red.*
</details>

### Exercise 5 (Frozen Lake, solved by planning)
Gymnasium's Frozen Lake publishes its model: `lake.unwrapped.P[state][action]` is a list of `(probability, next_state, reward, terminated)`, the same format as our `transitions`. Run value iteration on the slippery 8 × 8 lake (`gym.make("FrozenLake8x8-v1", is_slippery=True)`) with gamma 0.99, then play 1,000 episodes with the resulting policy and report how often it reaches the present. Record one episode as a video.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
import gymnasium as gym

lake = gym.make("FrozenLake8x8-v1", is_slippery=True, render_mode="rgb_array")
P = lake.unwrapped.P
n_states, n_actions = lake.observation_space.n, lake.action_space.n

def lake_action_values(V, state, gamma):
    q = np.zeros(n_actions)
    for action in range(n_actions):
        for probability, next_state, reward, terminated in P[state][action]:
            q[action] += probability * (reward + gamma * (0.0 if terminated else V[next_state]))
    return q

V = np.zeros(n_states)
for sweep in range(2000):
    new_V = np.array([lake_action_values(V, s, 0.99).max() for s in range(n_states)])
    done = np.abs(new_V - V).max() < 1e-8
    V = new_V
    if done:
        break
lake_policy = [int(lake_action_values(V, s, 0.99).argmax()) for s in range(n_states)]
print(f"converged after {sweep + 1} sweeps")

wins, frames = 0, []
for episode in range(1000):
    state, info = lake.reset(seed=episode)
    terminated = truncated = False
    while not (terminated or truncated):
        if episode == 0:
            frames.append(lake.render())
        state, reward, terminated, truncated, info = lake.step(lake_policy[state])
    wins += reward
print(f"reached the present in {wins / 10:.0f}% of 1,000 episodes")
show_video(frames, fps=8, width=360)
```

*On this lake the character slides in an unintended direction two times out of three, so even the optimal policy fails sometimes, and an episode is cut off after 200 steps. Watch the video: the policy often points the character at a wall or away from a hole, so that every possible slide is harmless.*
</details>

### Exercise 6 (Evaluating without sweeping (a little harder))
The Bellman expectation equation is a system of linear equations, $v = r_\pi + \gamma P_\pi v$, where $P_\pi$ is the matrix of transition probabilities under the policy and $r_\pi$ the vector of expected immediate rewards. Build $P_\pi$ and $r_\pi$ for the random policy on the dry room and solve the system directly with `np.linalg.solve`. Compare with `V_random`.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
env = GridWorld()
policy = np.full((env.n_states, 4), 0.25)
P_pi = np.zeros((env.n_states, env.n_states))
r_pi = np.zeros(env.n_states)

for state in usable_states(env):
    for action in range(4):
        for probability, next_state, reward, terminated in env.transitions(state, action):
            weight = policy[state, action] * probability
            r_pi[state] += weight * reward
            if not terminated:                         # terminal states contribute no future value
                P_pi[state, next_state] += weight

V_exact = np.linalg.solve(np.eye(env.n_states) - GAMMA * P_pi, r_pi)
print(f"start state, solved directly: {V_exact[0]:.4f}")
print(f"start state, by sweeping:     {V_random[0]:.4f}")
print(f"largest difference over all states: {np.abs(V_exact - V_random).max():.6f}")
```

*Rearranged, the equation reads $(I - \gamma P_\pi)\,v = r_\pi$. Solving it directly is fine for 25 states. The cost grows with the cube of the number of states, which is why iterative sweeps are preferred for anything large.*
</details>

## 🔑 Key takeaways

- $v_\pi(s)$ is the expected return from state $s$ under policy $\pi$; $q_\pi(s, a)$ is the same after first taking action $a$.
- The **Bellman expectation equation** links each state's value to its successors' values: reward now, plus the discounted value of what comes next.
- **Policy evaluation** computes $v_\pi$ by repeated sweeps; **policy improvement** acts greedily on those values and never makes things worse.
- **Policy iteration** alternates the two; **value iteration** uses the **Bellman optimality equation**, with a maximum in place of the average. Both reach the optimal policy.
- The optimal policy depends on the **probabilities**, the **rewards** and the **discount factor**: change any of them and the best behaviour changes.
- Dynamic programming needs the **model** and a sweep over **all states**. The rest of the course removes both requirements.

---
**Next:** *Notebook 04: Learning from Complete Episodes*, where the rules are hidden and the agent can only play and observe.

---
*© Ioannis Tsioulis. *From Random Moves to Winning Agents: Reinforcement Learning with Games*. Prepared for educational use.*